# 다중공선성 분석 (VIF)
- 대상 파일: `Membership_v2_with_derived_features.csv`
- VIF > 10: 다중공선성 심각, inf: 완전 선형 결합

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_csv('Membership_v2_with_derived_features.csv')

# 수치형 컬럼만 선택
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
X = df[num_cols].dropna()

# 분산 0인 컬럼 제거 (상수 컬럼)
X = X.loc[:, X.var() > 0]

print(f'분석 대상 컬럼 수: {X.shape[1]}')
X.head(3)

In [ ]:
vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)

vif_data

In [ ]:
# 구간별 요약
inf_cols  = vif_data[vif_data['VIF'] == np.inf]
high_cols = vif_data[(vif_data['VIF'] > 10) & (vif_data['VIF'] != np.inf)]
ok_cols   = vif_data[vif_data['VIF'] <= 10]

print(f'=== inf (완전 다중공선성): {len(inf_cols)}개 ===')
print(inf_cols['feature'].tolist())

print(f'\n=== VIF > 10 (심각): {len(high_cols)}개 ===')
print(high_cols[['feature', 'VIF']].to_string(index=False))

print(f'\n=== VIF <= 10 (양호): {len(ok_cols)}개 ===')
print(ok_cols[['feature', 'VIF']].to_string(index=False))

---
# 상관관계 분석

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rc

rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

corr = X.corr()

# 전체 히트맵
fig, ax = plt.subplots(figsize=(30, 26))
sns.heatmap(
    corr,
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    annot=False,
    linewidths=0.3,
    ax=ax
)
ax.set_title('전체 변수 상관관계 히트맵', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# |상관계수| > 0.8인 변수 쌍 추출
threshold = 0.8

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={'level_0': 'var1', 'level_1': 'var2', 0: 'corr'})
)
high_corr_pairs = high_corr_pairs[high_corr_pairs['corr'].abs() >= threshold]
high_corr_pairs = high_corr_pairs.reindex(
    high_corr_pairs['corr'].abs().sort_values(ascending=False).index
).reset_index(drop=True)

print(f'|상관계수| >= {threshold} 변수 쌍: {len(high_corr_pairs)}개')
high_corr_pairs

In [ ]:
# 높은 상관관계 변수만 모아서 히트맵 시각화
high_vars = list(set(high_corr_pairs['var1'].tolist() + high_corr_pairs['var2'].tolist()))
corr_sub = corr.loc[high_vars, high_vars]

fig, ax = plt.subplots(figsize=(max(10, len(high_vars) * 0.5), max(8, len(high_vars) * 0.45)))
sns.heatmap(
    corr_sub,
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    ax=ax
)
ax.set_title(f'|상관계수| >= {threshold} 변수 히트맵', fontsize=14)
plt.tight_layout()
plt.show()

## 조치 필요 목록

| 문제 | 해당 변수 | 조치 |
|------|-----------|------|
| `watch_days / 21 = active_ratio` | `watch_days`, `active_ratio` | 둘 중 하나 제거 |
| `w1+w2+w3 = total_watch_time` | `watch_time(min)_w1/w2/w3`, `total_watch_time(min)` | total 또는 주차별 중 선택 |
| `w1+w2+w3 = total_watch_count` | `watch_session_w1/w2/w3`, `total_watch_count` | 동일 |
| diff 선형 결합 | `diff_between_w2_w1`, `diff_between_w3_w1`, `diff_between_w3_w2` | 최대 2개만 유지 |
| 더미 완전 분리 | `reg_hour_morning/afternoon/evening/night` | 기준 범주 1개 제거 |
| 원본+파생 중복 | `max_screen`, `is_standard`, `is_premium` | `max_screen` 제거 또는 더미 중 하나 제거 |
| 장르 비율 합=1 | `action_adventure_ratio` ... `other_ratio` | `other_ratio` 제거 |